# 384-Well Plate: Data Import (Primary Analysis)

This notebook loads the raw plate-reader Excel files for the 384-well primary experiment, reshapes them into tidy long format, attaches experimental metadata, and saves the combined result as a single CSV for downstream analysis.

This experiment includes 12 wines (W1-W12), 14 sensors, with both the peptide identity and metal/dye chemistry varying across sensors(Cu-PCV, Cu-CAS, Ni-BPR). All 14 sensors and 4 wines fit on a single 384-well plate, so three plates cover all 12 wines (4 wines each), rather than the 6 plates needed for the 96-well pilot's 6 wines.

load_plate, tidy_plate, and add_plate_map are reused unchanged from 01_96well_Import.ipynb. Only the plate map construction differs, because the physical layouts are different. This is the primary dataset and the results here are the main analysis, with the 96-well pilot compared qualitatively rather than merged.

## Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Loading functions

Identical to 01_96well_Import.ipynb, the raw plate-reader file format doesn't depend on plate size.

In [ ]:
#Loads one raw plate-reader Excel file and returns a wavelength x well absorbance table
#The header row isn't at a fixed positiion across files, so it's located by searching for the "Wavel." label

def load_plate(file_path):

    #Read the first 100 rows to find the header
    preview = pd.read_excel(
        file_path,
        header = None,
        nrows = 100
    )

    #Find the row containing Wavel.
    matches = preview.eq("Wavel.")

    if not matches.any().any():
        raise ValueError(f"Could not find 'Wavel.' in {file_path}")

    header_row = matches.any(axis = 1).idxmax()

    #Read the data
    df = pd.read_excel(
        file_path,
        header = header_row,
        nrows = 226
    )

    #Rename the wavelength column
    df = df.rename(columns = {"Wavel.": "Wavelength"})

    #Ensure numeric
    df["Wavelength"] = pd.to_numeric(df["Wavelength"])

    return df

In [ ]:
#Reshapes one plate from wide (one column per well) to long (one row per wavelength x well measurement)
#Each row is tagged with which plate file it came from

def tidy_plate(df, plate_name):

    df_long = df.melt(
        id_vars = "Wavelength",
        var_name = "Well",
        value_name = "Absorbance"
    )

    #Keep track of which plate the data came from
    df_long["Plate"] = plate_name

    return df_long

## 384-well plate map

The 384-well plate has 16 rows (A-P) and 24 columns. Rows B-O each correspond to one of the 14 fixed sensors (in a set order). Rows A and P are wine-only controls (wine present, no sensor) rather than the 96-well design's buffer-only (HEPES) control.h is is why baseline correction is done differently between the two datasets. Columns are split into four 6-column blocks, one per wine, giving 6 replicates per wine per sensor on a single plate.

In [ ]:
#Builds the well (peptide, sensor, wine, replicate) mapping for one 384-well plate
#All 14 sensors fit on a single plate, so there's no peptide_start parameter needed here

def create_plate_map_384(wine_names = ("W1", "W2", "W3", "W4")):

    rows = list("ABCDEFGHIJKLMNOP")
    columns = range(1, 25)

    #Sensor identities for rows B-O, in order
    sensors = [
        "Cu-IHIGHHI-PCV", "Cu-WEEHEE-PCV", "Cu-WAHEDEFF-PCV", "Cu-PHGGGWGQ-PCV",
        "Cu-FHFPHHF-PCV", "Cu-WHCCHDHCD-PCV", "Cu-WGHGGHHG-PCV", "Cu-FHFPHHF-CAS",
        "Cu-WDHHHD-PCV", "Cu-WEEHEE-CAS", "Cu-WEHHHE-PCV", "Ni-FHFPHHF-BPR",
        "Cu-WDDHDD-PCV", "Ni-WEEHEE-BPR"
    ]

    plate_map = []

    for row_index, row in enumerate(rows):

        #Row A and Row P are wine control (no peptide)
        if row_index == 0 or row_index == len(rows) - 1:
            peptide = None
            sensor = "Wine control"
        else:
            peptide = row_index
            sensor = sensors[row_index - 1]

        for column in columns:
            well = f"{row}{column}"

            #Wine 1
            if column <= 6:
                sample = wine_names[0]
                replicate = column

            #Wine 2
            elif column <= 12:
                sample = wine_names[1]
                replicate = column - 6

            #Wine 3
            elif column <= 18:
                sample = wine_names[2]
                replicate = column - 12

            #Wine 4
            else:
                sample = wine_names[3]
                replicate = column - 18

            plate_map.append({
                "Well": well,
                "Peptide": peptide,
                "Sensor": sensor,
                "Wine": sample,
                "Replicate": replicate
            })

    return pd.DataFrame(plate_map)

In [ ]:
#Attaches experimental metadata to a tidy plate dataframe by joining on well

def add_plate_map(df_long, plate_map):

    df = df_long.merge(
        plate_map,
        on = "Well",
        how = "left"
    )

    return df

In [ ]:
#Run one raw plate file through the full load, tidy, map pipeline in one call

def process_plate_384(
            file_path,
            plate_name,
            wine_names
        ):

    plate = load_plate(file_path)

    plate_long = tidy_plate(plate, plate_name)

    plate_map = create_plate_map_384(wine_names)

    plate_complete = add_plate_map(plate_long, plate_map)

    return plate_complete

## Plate registry

One entry per  plate file. Each plate covers 4 wines, so 3 plates cover all 12 wines (compared to 6 plates for the 96-well pilot's 6 wines). 

In [7]:
plates_384 = [
    {
        "file_path": "../../data/raw/Wines1_4_384well.xlsx",
        "plate_name": "W1_4_384",
        "wine_names": ("W1", "W2", "W3", "W4")
    },
    {
        "file_path": "../../data/raw/Wines5_8_384well.xlsx",
        "plate_name": "W5_8_384",
        "wine_names": ("W5", "W6", "W7", "W8")
    },
    {
        "file_path": "../../data/raw/Wines9_12_384well.xlsx",
        "plate_name": "W9_12_384",
        "wine_names": ("W9", "W10", "W11", "W12")
    }
]

## Run the pipeline

Same try/except-per-plate pattern as the 96-well notebook, so one bad file doesn't stop the whole run.

In [8]:
all_plates_384 = []

for plate in plates_384:
    try:
        print(f"Processing {plate['plate_name']}...")
        df = process_plate_384(**plate)
        all_plates_384.append(df)
        print("Success")

    except Exception as e:
        print(f"Failed: {plate['plate_name']}")
        print(e)
        print()

master_df_384 = pd.concat(all_plates_384, ignore_index = True)

print("Done!")
print(master_df_384.shape)

Processing W1_4_384...
Success
Processing W5_8_384...
Success
Processing W9_12_384...
Success
Done!
(260352, 8)


## Quality check

Confirms each plate has the expected number of wells (384), the full
12 wines are all present, all 14 sensors (+ wine control) are captured,
and replicate counts are even across the 1-6 range.

In [ ]:
print(master_df_384["Wine"].unique())          
print(master_df_384["Sensor"].nunique())          
print(master_df_384["Replicate"].value_counts())  
print(master_df_384["Plate"].value_counts())         

['W1' 'W2' 'W3' 'W4' 'W5' 'W6' 'W7' 'W8' 'W9' 'W10' 'W11' 'W12']
15
Replicate
1    43392
2    43392
3    43392
4    43392
5    43392
6    43392
Name: count, dtype: int64
Plate
W1_4_384     86784
W5_8_384     86784
W9_12_384    86784
Name: count, dtype: int64


## Save

In [10]:
master_df_384.to_csv(
    "../../data/processed/master_df_384.csv",
    index = False
)